## Part B — fresh process

**Restart the runtime first** (Runtime → Restart session). The cells below read only files under
`/content`, so they run in a new Python process: that is the reload test the spec asks for, and it frees
the GPU (a 4B model twice does not fit a 15 GB T4).

In [ ]:
# ---- B1: reload the exported adapter in this fresh process -------------------------------------
import json, sys, os, math, gc, time, shutil, torch
PART_A = json.load(open("/content/part_a.json"))
B, EXP, MODE, SEED = PART_A["bundle_dir"], PART_A["export_dir"], PART_A["mode"], PART_A["seed"]
sys.path.insert(0, B)
assert torch.cuda.memory_allocated() == 0, "not a fresh process: restart the runtime first"
from unsloth import FastLanguageModel
from training import colab_helpers as H, prepare as P
m, proc = FastLanguageModel.from_pretrained(model_name=EXP, max_seq_length=P.MAX_TOKENS,
                                            load_in_4bit=False, load_in_16bit=True)
tok = getattr(proc, "tokenizer", proc)
FastLanguageModel.for_inference(m)
sample = json.load(open("/content/reload_sample.json"))
got = [H.generate_label(m, tok, x["messages"]) for x in sample]
RELOAD = {"reload_match": sum(g == x["expect"] for g, x in zip(got, sample)), "of": len(sample),
          "fingerprint_match": P.sha256(tok.chat_template) == PART_A["fingerprint"]["chat_template_sha256"],
          "weights_dtype": str(next(m.parameters()).dtype)}
print(RELOAD)
del m; gc.collect(); torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.10.5: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


`causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


{'reload_match': 5, 'of': 5, 'fingerprint_match': True, 'weights_dtype': 'torch.float16'}


In [ ]:
# ---- B2: continuation test (smoke only): 20 steps straight vs 10 + save/reload + 10 ----------
RESUME = None
if MODE == "smoke" or PART_A.get("dry_run_steps"):
    from transformers import Trainer, TrainerCallback
    from datasets import Dataset

    class StopAt(TrainerCallback):
        def __init__(self, n): self.n = n
        def on_step_end(self, args, state, control, **kw):
            if state.global_step >= self.n: control.should_training_stop = True
            return control

    def run(out, stop=None, resume=None):
        m2, proc2 = FastLanguageModel.from_pretrained(model_name=PART_A["model_id"], revision=PART_A["revision"],
                                                      max_seq_length=P.MAX_TOKENS, load_in_4bit=False, load_in_16bit=True)
        tok2 = getattr(proc2, "tokenizer", proc2)
        m2 = FastLanguageModel.get_peft_model(m2, **H.LORA, random_state=SEED, max_seq_length=P.MAX_TOKENS)
        rows = H.encode(tok2, H.load_records(B, PART_A["train_file"])[:20 * 16])
        t = Trainer(model=m2, args=H.make_args(out, 20, 10, smoke=True, seed=SEED, evaluate=False),
                    train_dataset=Dataset.from_list(rows), data_collator=H.make_collate(tok2.pad_token_id),
                    callbacks=[StopAt(stop)] if stop else [])
        t.train(resume_from_checkpoint=resume)
        losses = {h["step"]: h["loss"] for h in t.state.log_history if "loss" in h}
        del t, m2; gc.collect(); torch.cuda.empty_cache()
        return losses
    full = run("/content/rt_full")
    run("/content/rt_half", stop=10)
    cont = run("/content/rt_half", resume="/content/rt_half/checkpoint-10")
    steps = [s for s in range(11, 21) if s in full and s in cont]
    diff = max(abs(full[s] - cont[s]) for s in steps)
    RESUME = {"steps_compared": steps, "uninterrupted": [full[s] for s in steps],
              "continued": [cont[s] for s in steps], "max_abs_loss_diff": diff,
              "tolerance": 0.05, "pass": bool(steps) and diff <= 0.05}
    print(json.dumps(RESUME, indent=1))

==((====))==  Unsloth 2026.10.5: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 320 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


...

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 320 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)


Step,Training Loss
11,0.000297
12,0.000030
13,0.000022
14,0.000018
15,0.000013
16,0.000011
17,0.000009
18,0.000009
19,0.000007
20,0.000006


{
 "steps_compared": [
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20
 ],
 "uninterrupted": [
  0.0002965251333080232,
  2.9744642233708873e-05,
  2.171269807149656e-05,
  1.8293780158273876e-05,
  1.3464509720506612e-05,
  1.0920946806436405e-05,
  9.351881089969538e-06,
  9.090357707464136e-06,
  7.432643997162813e-06,
  6.385108008544194e-06
 ],
 "continued": [
  0.0002965251333080232,
  2.9744642233708873e-05,
  2.171269807149656e-05,
  1.8293780158273876e-05,
  1.3464509720506612e-05,
  1.0920946806436405e-05,
  9.351881089969538e-06,
  9.090357707464136e-06,
  7.432643997162813e-06,
  6.385108008544194e-06
 ],
 "max_abs_loss_diff": 0.0,
 "tolerance": 0.05,
 "pass": true
}


In [ ]:
# ---- B3: smoke report (the evidence for 'Notebook smoke-tested') -------------------------------
REPORT = {**PART_A, "reload": RELOAD, "resume_test": RESUME}
REPORT["passed"] = bool(PART_A["train"]["global_steps"] >= 20
                        and all(math.isfinite(l) for _, l in PART_A["train"]["losses"])
                        and RELOAD["reload_match"] == RELOAD["of"] and RELOAD["fingerprint_match"]
                        and (RESUME is None or RESUME["pass"]))
name = ("gpu_smoke_test.json" if MODE == "smoke" else f"dryrun_{MODE}_report.json" if PART_A.get("dry_run_steps")
        else f"train_{MODE}_report.json")
json.dump(REPORT, open(f"/content/{name}", "w"), indent=1, default=str)
shutil.make_archive(f"/content/export_{MODE}", "zip", EXP)
print("passed:", REPORT["passed"], "->", name, "(download it into reports/)")

passed: True -> dryrun_fast_report.json (download it into reports/)
